# 19.08 - R3D-18 multi-clip evaluation

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Multi-clip latency/metric benchmark and submission predictions.

Practice deterministic temporal coverage and video-level logit aggregation using prepared clip logits, avoiding repeated heavy model inference while preserving the evaluation boundary.

## Core Ideas

Training can sample random clips, but validation should use deterministic positions. Aggregate either logits or probabilities consistently, never mix them across runs. Split at video level to prevent clip leakage. More clips increase temporal coverage and latency, so compare one, three, and five clips on the same videos.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

SEED = 19
np.random.seed(SEED)
torch.manual_seed(SEED)

## Prepared Video-Level Clip Logits

Six videos have five ordered clip logits each and balanced labels. These cached outputs stand in for one R3D-18 forward pass per clip.

In [ ]:
video_ids = [f"video_{index:02d}" for index in range(6)]
video_labels = torch.tensor([0, 0, 1, 1, 2, 2])
clip_logits = torch.randn(6, 5, 3) * 0.25
for video_index, label in enumerate(video_labels):
    clip_logits[video_index, :, label] += torch.tensor([0.4, 0.8, 1.0, 0.8, 0.4])
print("cached logits:", clip_logits.shape, "support:", torch.bincount(video_labels).tolist())

## Exercise 19-A: Choose deterministic clip indices

Select evenly spaced indices including temporal endpoints when more than one clip is requested.

**Return structure — `uniform_clip_indices`:** A sorted `list[int]` of length `min(requested,total_clips)`, with unique values in `[0,total_clips-1]`.

In [ ]:
# TODO 19-A
def uniform_clip_indices(total_clips, requested):
    raise NotImplementedError("Complete Exercise 19-A")


# Smoke check: inspect one-, three-, and five-clip coverage.
clip_selections = {count: uniform_clip_indices(5, count) for count in [1, 3, 5]}
print(clip_selections)

## Exercise 19-B: Aggregate video logits

Average selected clip logits for each video and return video-level predictions.

**Return structure — `aggregate_video_logits`:** A tuple `(mean_logits, predictions)`: CPU float32 `[V,C]` and CPU int64 `[V]`.

In [ ]:
# TODO 19-B
def aggregate_video_logits(all_clip_logits, selected_indices):
    raise NotImplementedError("Complete Exercise 19-B")


# Smoke check: aggregate the three-clip setting.
three_clip_mean, three_clip_predictions = aggregate_video_logits(clip_logits, clip_selections[3])
print(three_clip_mean.shape, three_clip_predictions.tolist())

## Exercise 19-C: Benchmark clip counts

Report Macro-F1 and estimated latency with one shared per-clip latency assumption.

**Return structure — `multi_clip_benchmark`:** A three-row DataFrame with columns `clip_count`, `selected_indices`, `video_count`, `class_support`, `macro_f1`, and `estimated_latency_ms`.

In [ ]:
# TODO 19-C
def multi_clip_benchmark(all_clip_logits, labels, clip_counts=(1, 3, 5), latency_per_clip_ms=12.0):
    raise NotImplementedError("Complete Exercise 19-C")


# Smoke check and full-video evidence.
multi_clip_evidence = multi_clip_benchmark(clip_logits, video_labels)
print(multi_clip_evidence.to_string(index=False))

## Exercise 19-D: Create submission-ready predictions

Preserve test/video ID order and calculate confidence from aggregated probabilities.

**Return structure — `video_prediction_rows`:** A `list[dict]` of length `V`; every row has `video_id` (`str`), `prediction` (`int`), `confidence` (`float`), and `clip_count` (`int`).

In [ ]:
# TODO 19-D
def video_prediction_rows(ids, all_clip_logits, selected_indices):
    raise NotImplementedError("Complete Exercise 19-D")


# Smoke check: create five-clip output rows.
submission_rows = video_prediction_rows(video_ids, clip_logits, clip_selections[5])
print(pd.DataFrame(submission_rows).to_string(index=False))

## Test Cases

**Return structure — `run_day19_tests`:** Returns `None`; assertions and `Day 19 tests passed` communicate success.

In [ ]:
def run_day19_tests():
    assert clip_selections == {1: [0], 3: [0, 2, 4], 5: [0, 1, 2, 3, 4]}
    assert three_clip_mean.shape == (6, 3) and three_clip_predictions.shape == (6,)
    assert multi_clip_evidence["clip_count"].tolist() == [1, 3, 5]
    assert all(value == [2, 2, 2] for value in multi_clip_evidence["class_support"])
    assert multi_clip_evidence["estimated_latency_ms"].tolist() == [12.0, 36.0, 60.0]
    assert len(submission_rows) == 6 and [row["video_id"] for row in submission_rows] == video_ids
    assert all(row["clip_count"] == 5 and 0.0 <= row["confidence"] <= 1.0 for row in submission_rows)
    print("Day 19 tests passed")


run_day19_tests()

## Day 19 Checklist

- [ ] Select deterministic validation clips.
- [ ] Aggregate one representation consistently.
- [ ] Compare coverage and latency on identical videos.
- [ ] Preserve stable video ID order.
- [ ] Run the test cases.